# Entrenar la palabra de activación **"hey paco"** (openWakeWord)

Ejecuta este cuaderno en **Google Colab** con GPU (`Entorno de ejecución → Cambiar tipo → T4 GPU`).
Tarda ~20-40 min. Al final descargas `hey_paco.tflite` y `hey_paco.onnx`.

No hace falta grabar nada: las muestras positivas son voz sintética (Piper) en español.

In [ ]:
# 1. Dependencias
!pip install -q openwakeword torch torchinfo torchmetrics speechbrain audiomentations \
    'numpy<2' onnx onnx-tf tensorflow-cpu mutagen acoustics pronouncing
!git clone -q https://github.com/dscripka/openWakeWord.git
!git clone -q https://github.com/rhasspy/piper-sample-generator.git
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
!wget -q -O piper-sample-generator/models/es_ES-glados.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/es_ES-glados.pt || true
print('ok')

In [ ]:
# 2. Datos de fondo (ruido + reverberación) y características negativas precalculadas
import os
os.makedirs('data', exist_ok=True)
# RIRs (reverberación) y ruido
!wget -q -O data/mit_rirs.zip https://mcdermottlab.mit.edu/Reverb/IR_Survey.zip || true
!cd data && (unzip -q -o mit_rirs.zip -d mit_rirs || true)
# Características negativas de openWakeWord (~2 GB): habla y ruido que NO es la palabra
from huggingface_hub import hf_hub_download
for f in ['openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
          'validation_set_features.npy']:
    p = hf_hub_download('davidscripka/openwakeword_features', f, local_dir='data')
    print(p)

In [ ]:
# 3. Config de entrenamiento para "hey paco"
cfg = '''
target_phrase: ["hey paco", "oye paco", "ey paco"]
model_name: hey_paco
n_samples: 30000
n_samples_val: 2000

tts_batch_size: 50
piper_sample_generator_path: ./piper-sample-generator
tts_model_path: ./piper-sample-generator/models/es_ES-glados.pt

output_dir: ./hey_paco_out
rir_paths: [./data/mit_rirs]
background_paths: []
false_positive_validation_data_path: ./data/validation_set_features.npy
feature_data_files: {ACAV100M_sample: ./data/openwakeword_features_ACAV100M_2000_hrs_16bit.npy}

batch_n_per_class: {ACAV100M_sample: 1024, adversarial_negative: 50, positive: 50}
steps: 50000
max_negative_weight: 1500
target_accuracy: 0.7
target_recall: 0.5
'''
open('hey_paco.yaml','w').write(cfg)
print(cfg)

In [ ]:
# 4. Generar muestras + entrenar  (esto es lo que tarda)
%cd openWakeWord
!python openwakeword/train.py --training_config ../hey_paco.yaml --generate_clips
!python openwakeword/train.py --training_config ../hey_paco.yaml --augment_clips
!python openwakeword/train.py --training_config ../hey_paco.yaml --train_model
%cd ..

In [ ]:
# 5. Descargar el modelo
from google.colab import files
import glob
for f in glob.glob('hey_paco_out/hey_paco.*'):
    if f.endswith(('.onnx', '.tflite')):
        print(f); files.download(f)

## En el asistente

```bash
mkdir -p ~/Documentos/asitente\ de\ voz/models
# copia hey_paco.onnx (PC) y hey_paco.tflite (Raspberry) ahí
```

`config.yaml`:
```yaml
wakeword:
  provider: openwakeword
  model: /home/alejandro/Documentos/asitente de voz/models/hey_paco.onnx
  threshold: 0.5
  framework: onnx      # tflite en la Raspberry
```